## Project Intent

In this notebook, we'll explore how to generate synthetic auto insurance test data using a Streamlit interface and OpenAI's GPT-4o-mini model. At the end, we'll launch the app for a live demo.

As a software engineer at a leading insurance company, I collaborate closely with pricing and underwriting modelers and actuaries to build systems that deliver accurate rates to market. This project is particularly compelling because it showcases how generative AI—when fine‑tuned on an organization’s rating models, algorithms, and input schemas—can dramatically reduce the complexity of creating comprehensive test datasets. By automating the generation of structured policy, driver, and incident data, we not only accelerate our development cycles but also improve test coverage across diverse risk scenarios.

## Technology Stack

- **Python**: Core language for data handling and backend logic
- **Streamlit**: Rapid UI prototyping for web interfaces
- **OpenAI API (GPT-4o-mini)**: Generative model for synthetic data
- **JSON**: Data parsing and preview
- **Podman/Docker**: Containerization for portability

## Model Fine-Tuning

To improve output consistency and ensure adherence to our insurance schema, we fine-tuned the base **gpt-4o-mini** model on a custom dataset of JSON examples.  Given the academic nature of this assignment, a key consideration was cost.  Therefore, we limited the scope of the training data, number of training epochs, and selected a low-cost base model for tuning to demonstrate understanding while keeping costs low.  

**Dataset:**
- ~40 JSONL records covering policy scenarios (teen drivers, accidents, multiple vehicles, etc.)
- Each JSONL entry is a `messages` array of `{role, content}` objects—covering system, user, and assistant turns—so that OpenAI’s chat fine-tuner can ingest it directly.

**Training Details:**
- Model: gpt-4o-mini
- Epochs: 1
- Learning rate: 1e-5
- Batch size: 1
- Validation split: 10% (default)

After fine-tuning, the model exhibits:
- Higher schema adherence (required fields present)
- Reduced malformed JSON outputs
- Faster convergences during streaming inference

## Inspecting the Streamlit App Code
Let's take a quick look at the main app file (`ai-synth-data-app.py`) to understand the UI and logic.

In [1]:
from IPython.display import Code

# Display the contents of app.py for review
display(Code('ai-synth-data-app.py', language='python'))

import os
import re
import json
from dotenv import load_dotenv
import streamlit as st
from openai import OpenAI

# Load environment variables from .env
load_dotenv()

# Retrieve OpenAI API key
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    st.error("OPENAI_API_KEY not found in environment; please check your .env file.")
    st.stop()

# Initialize the new OpenAI v1 client
client = OpenAI(api_key=api_key)

# Page configuration
st.set_page_config(page_title="Auto Insurance Synthetic Data Generator", layout="wide")

st.title("Auto Insurance Synthetic Data Generator")
st.write(
    "Describe the policy data you need below and click 'Generate Data' button to obtain your synthetic data."
)

# User inputs
scenario = st.text_area(
    "Scenario Prompt",
    value="Generate a policy with a 16-year-old driver who has 1 at-fault accidents."
)
num_policies = st.slider("Number of Policies", min_value=1, max_value=20, value=1)
generate = st.button("Generate Data")

def extract_json_array(raw_text: str) -> str:
    """
    This function will remove markdown fences returned from the model and extract the JSON array block.
    """
    # Strip code fences
    text = re.sub(r"```(?:json)?\s*", "", raw_text)
    text = re.sub(r"\s*```", "", text)
    # Extract JSON array
    match = re.search(r"(\[.*\])", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON array found in the model output.")
    return match.group(1)

if generate:
    with st.spinner("Generating synthetic data..."):
        prompt = f"{scenario}\n\nReturn ONLY the JSON array of {num_policies} policies."
        response = client.chat.completions.create(
            model="ft:gpt-4o-mini-2024-07-18:darkwatercdr:dsc670-week09-pricing-ft:BWFyEa5u",
            messages=[
                {"role": "system", "content": "You are a JSON-only data generator for auto insurance policies."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            top_p=1.0
        )
        raw_output = response.choices[0].message.content

        try:
            json_text = extract_json_array(raw_output)
            data = json.loads(json_text)
            
            st.subheader("Generated Policies JSON")
            st.json(data)
            
            st.download_button(
                label="Download JSON",
                data=json.dumps(data, indent=2),
                file_name="synthetic_policies.json",
                mime="application/json"
            )

        except Exception as e:
            st.error(f"Failed to parse JSON: {e}")
            st.code(raw_output, language="text")

## Running the Streamlit App
Finally, we'll launch the Streamlit app. Once it starts, use the browser UI
to submit a data generation request and see the output.

**Note:** The notebook environment may not support running Streamlit apps directly. If you encounter issues, consider running the app from your local terminal or command prompt using:

```bash
.aienv\Scripts\Activate.ps1 # Activate the virtual environment in PowerShell
streamlit run --server.showEmailPrompt false ai-synth-data-app.py
```

In [2]:
# Run the Streamlit app
# Should open a browser window
# if not, it can be accessed at http://localhost:8501

# Note:  depending on your version of python and dependencies (e.g., Jupyter, Streamlit),
#        you may need to stop this cell manually to regain control of the notebook.
get_ipython().system('streamlit run --server.showEmailPrompt false ai-synth-data-app.py')

^C



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8513
  Network URL: http://172.29.7.100:8513



## Conclusion
 
Throughout this project, we successfully fine‑tuned OpenAI’s **gpt‑4o‑mini** model to serve as a reliable JSON‑structured synthetic data generator for auto insurance pricing and underwriting workflows. By creating and tuning on a representative JSONL dataset, we achieved consistent schema adherence and minimized malformed outputs. Integrating this tuned model into a Streamlit app demonstrates an end‑to‑end use case: from prompt-based data generation to immediate downstream consumption.
 
That said, our initial training set was limited in size and variety, which constrains the model’s ability to handle edge‑case scenarios—particularly those involving nuanced factors like driver age thresholds. To improve accuracy, our next steps include:
- Expanding the training dataset with a broader mix of policy examples (e.g., multi‑driver households, complex claims histories).
- Increasing the number of training Epochs and/or selecting a model based on effectiveness instead of optimized for cost.
- Exploring alternative fine‑tuning strategies, such as incorporating insurer‑specific rate manuals or underwriting guidelines directly into the training corpus.
- Evaluating incremental tuning approaches: first build a deep understanding of rating rules, then refine for strict JSON output.
 
This notebook illustrates both the mechanics of fine‑tuning and its application within a user‑friendly interface. While there remains room for enhanced precision, the project confirms our ability to train, deploy, and operate a generative AI pipeline for structured insurance data generation—key skills for applying AI in actuarial and GenAI‑Ops contexts.